In [1]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [2]:
from torchvision import datasets

train_dataset = datasets.ImageFolder(
    root="data/intel/seg_train",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root="data/intel/seg_test",
    transform=transform
)

In [3]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [4]:
print(train_dataset.classes)
print(len(train_dataset), len(test_dataset))

['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
14034 3000


Resnet transfer learning


In [5]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=True)

model.fc = nn.Linear(model.fc.in_features, 6)  # 6 classes
model = model.to(device)

c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [9]:
for epoch in range(5):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if i % 100 == 0:
            print(f"Epoch {epoch+1}, Batch {i}, Loss: {loss.item():.4f}")

    train_acc = correct / total

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Train Acc: {train_acc:.4f}")

Epoch 1, Batch 0, Loss: 0.0225
Epoch 1, Batch 100, Loss: 0.0069
Epoch 1, Batch 200, Loss: 0.0043
Epoch 1, Batch 300, Loss: 0.0334
Epoch 1, Batch 400, Loss: 0.0219
Epoch 1, Batch 500, Loss: 1.0890
Epoch 1, Batch 600, Loss: 0.0240
Epoch 1, Batch 700, Loss: 0.1183
Epoch 1, Batch 800, Loss: 0.0651
Epoch 1, Loss: 0.0457, Train Acc: 0.9857
Epoch 2, Batch 0, Loss: 0.0039
Epoch 2, Batch 100, Loss: 0.0077
Epoch 2, Batch 200, Loss: 0.0026
Epoch 2, Batch 300, Loss: 0.0182
Epoch 2, Batch 400, Loss: 0.0087
Epoch 2, Batch 500, Loss: 0.0566
Epoch 2, Batch 600, Loss: 0.0076
Epoch 2, Batch 700, Loss: 0.0036
Epoch 2, Batch 800, Loss: 0.2168
Epoch 2, Loss: 0.0500, Train Acc: 0.9837
Epoch 3, Batch 0, Loss: 0.0095
Epoch 3, Batch 100, Loss: 0.0754
Epoch 3, Batch 200, Loss: 0.0414
Epoch 3, Batch 300, Loss: 0.0043
Epoch 3, Batch 400, Loss: 0.0227
Epoch 3, Batch 500, Loss: 0.0069
Epoch 3, Batch 600, Loss: 0.0192
Epoch 3, Batch 700, Loss: 0.0133
Epoch 3, Batch 800, Loss: 0.6800
Epoch 3, Loss: 0.0383, Train Acc:

In [10]:
#validation

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

val_acc = correct / total  
print(f"Validation Accuracy: {val_acc:.4f}")

Validation Accuracy: 0.9210
